In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df=spark.read.format("parquet")\
    .load("abfss://bronze@databricksetestorage.dfs.core.windows.net/products")
df.display()

In [0]:
df=df.drop("_rescued_data")
df.display()

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cat.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE 
LANGUAGE SQL
RETURN p_price*0.9

In [0]:
df.createOrReplaceTempView("products")

In [0]:
%sql
SELECT product_id,price,databricks_cat.bronze.discount_func(price) as discounted_price
FROM products

In [0]:
df=df.withColumn("discounted_price",expr("databricks_cat.bronze.discount_func(price)"))
df.display()

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cat.bronze.upper_func(brand STRING)
RETURNS STRING 
LANGUAGE PYTHON
AS 
$$
    return brand.upper()
$$

In [0]:
%sql
SELECT product_id,brand,databricks_cat.bronze.upper_func(brand) as brand_upper
FROM products

In [0]:
df.write.format("delta")\
    .mode("append")\
        .save("abfss://silver@databricksetestorage.dfs.core.windows.net/products")

In [0]:
%sql
create table if not exists databricks_cat.silver.products_silver
using delta
location "abfss://silver@databricksetestorage.dfs.core.windows.net/products"